In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score

df = pd.read_parquet("../data/processed/paysim_features.parquet")

if "week_bucket" not in df.columns:
    df["week_bucket"] = (df["step"] - df["step"].min()) // (24 * 7)
if "day_bucket" not in df.columns:
    df["day_bucket"] = (df["step"] - df["step"].min()) // 24

feature_columns = [
    "amount",
    "destination_transactions_last_24h",
    "destination_transactions_last_7d",
    "destination_avg_previous_amount",
    "destination_amount_deviation",
    "destination_is_first_transaction",
    "origin_balance_error",
    "destination_balance_error",
    "destination_balance_is_zero",
    "total_transactions",
    "total_transaction_amount",
    "avg_transaction_amount",
] + [c for c in df.columns if c.startswith("type_")]

dense_window = df[(df["day_bucket"] >= 5) & (df["day_bucket"] <= 16)]

def get_dense_period(days):
    subset = dense_window[dense_window["day_bucket"].isin(days)]
    return subset[feature_columns], subset["isFraud"]

def bootstrap_pr_auc_fast(model, X, y, n_bootstrap=200, seed=42):
    y_reset = y.reset_index(drop=True).values
    preds = model.predict_proba(X)[:, 1]
    rng = np.random.RandomState(seed)
    n = len(y_reset)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        y_sample = y_reset[idx]
        if y_sample.sum() == 0:
            continue
        pred_sample = preds[idx]
        scores.append(average_precision_score(y_sample, pred_sample))
    return np.array(scores)

print(f"Loaded {len(df):,} rows, {len(dense_window):,} dense-window rows.")

Loaded 6,362,620 rows, 4,918,825 dense-window rows.


In [2]:
X_naive, y_naive = get_dense_period([5, 6, 7, 8, 9, 10, 11, 12, 13])
X_delay, y_delay = get_dense_period([5, 6, 7, 8, 9, 10, 11])
X_test_delay, y_test_delay = get_dense_period([14, 15, 16])

print(f"Naive train:       {len(X_naive):,} rows, {y_naive.sum():,} fraud")
print(f"Delay-aware train: {len(X_delay):,} rows, {y_delay.sum():,} fraud")
print(f"Shared test:       {len(X_test_delay):,} rows, {y_test_delay.sum():,} fraud")

Naive train:       3,716,183 rows, 2,363 fraud
Delay-aware train: 2,889,724 rows, 1,875 fraud
Shared test:       1,202,642 rows, 822 fraud


In [3]:
lgb_naive = lgb.LGBMClassifier(
    n_estimators=200, num_leaves=15, max_depth=5,
    min_child_samples=100, reg_alpha=1.0, reg_lambda=1.0,
    scale_pos_weight=20, random_state=42, verbose=-1
)
lgb_naive.fit(X_naive, y_naive)

lgb_delay_aware = lgb.LGBMClassifier(
    n_estimators=200, num_leaves=15, max_depth=5,
    min_child_samples=100, reg_alpha=1.0, reg_lambda=1.0,
    scale_pos_weight=20, random_state=42, verbose=-1
)
lgb_delay_aware.fit(X_delay, y_delay)

print("Both models trained.")

Both models trained.


In [4]:
pr_auc_naive = average_precision_score(y_test_delay, lgb_naive.predict_proba(X_test_delay)[:, 1])
pr_auc_delay_aware = average_precision_score(y_test_delay, lgb_delay_aware.predict_proba(X_test_delay)[:, 1])

print(f"Naive model PR-AUC:       {pr_auc_naive:.4f}")
print(f"Delay-aware model PR-AUC: {pr_auc_delay_aware:.4f}")
print(f"Difference:               {pr_auc_delay_aware - pr_auc_naive:+.4f}")

Naive model PR-AUC:       0.8352
Delay-aware model PR-AUC: 0.7567
Difference:               -0.0785
